In [ ]:
# %% [Block 0 — Setup and Plotly template]
# Principle 5: define style once in a template; every chart inherits it.

import os
import numpy as np
import pandas as pd
import pyreadstat
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Shared palette (kept consistent across the course)
PALETTE = ['#7BB3B2', '#65A6BD', '#C997AF', '#B8B0D3', '#F4CF97', '#98B9A0', '#F6DECD']

# Build the template once
nso_template = go.layout.Template()
nso_template.layout = go.Layout(
    font=dict(family='Arial', size=13, color='#333'),
    title_font=dict(size=16, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    colorway=PALETTE,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=60, r=30, t=60, b=40),
)
pio.templates['nso'] = nso_template
pio.templates.default = 'nso'

# Plotly toolbar config (clean modebar, sensible PNG export)
PLOTLY_CONFIG = {
    'displaylogo': False,
    'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'autoScale2d'],
    'toImageButtonOptions': {'format': 'png', 'width': 1200, 'height': 700, 'scale': 2},
}

In [ ]:

ROOT_PATH = # your code here

# Load the three Census 2022 files (Persons ⋈ Households on QID where useful)
geography,  geo_meta    = pyreadstat.read_dta(os.path.join(ROOT_PATH, 'Census2022Geography.dta'), )
households, hh_meta     = pyreadstat.read_dta(os.path.join(ROOT_PATH, 'Census2022Households.dta'))
persons,    pp_meta     = pyreadstat.read_dta(os.path.join(ROOT_PATH, 'Census2022Persons.dta'))

for df, meta in zip([geography, households, persons], [geo_meta, hh_meta, pp_meta]):
    for col, labels in meta.variable_value_labels.items():
        df[col] = df[col].map(labels)

# Quick sanity check after loading:
# df.shape, df.head(), df.dtypes


# Make sure that following are of correct type! DERH_HHAGE,

In [ ]:
# %% [Block 1 — Read the data]
# Geography and Households are ONE row per household — read them fully.
# Persons is large (one row per person) — cap the read at 100,000 rows.
# Despite the files being Stata (.dta), we read everything with pandas.

ROOT_PATH =  # adjust to your local path

geography = pd.read_stata(f'{ROOT_PATH}/Census2022Geography.dta')
households = pd.read_stata(f'{ROOT_PATH}/Census2022Households.dta')

# Limit Persons to the first 100k rows with a chunked reader (avoids loading the lot).
with pd.read_stata(f'{ROOT_PATH}/Census2022Persons.dta', chunksize=100_000) as reader:
    persons = next(reader)


In [ ]:
hh = households
geo = geography

# --- Fix dtypes ---
# read_stata maps value labels to strings, so some genuinely numeric columns
# (age, household size, weights, year) arrive as categorical/object. Coerce them.
for c in ['DERH_HSIZE', 'DERH_HHAGE', 'HH_WGT']:
    hh[c] = pd.to_numeric(hh[c], errors='coerce')

for c in ['P04_AGE', 'P03_YEAR', 'P12B_YEARMOVED', 'PERS_WGT']:
    persons[c] = pd.to_numeric(persons[c], errors='coerce')

# --- Treat unlabelled sentinel codes as missing ---
persons.loc[persons['P12B_YEARMOVED'] == 8888, 'P12B_YEARMOVED'] = np.nan

print('Geography:', geo.shape)
print('Households:', hh.shape)
print('Persons (capped):', persons.shape)
hh.head(3)

In [ ]:
# %% [Task 1 — Age distribution with median reference]
# Principle 1: one trace (Histogram) + layout. Principle 4: the median line is layout chrome.

# ----- Step 1: Prepare -----


# ----- Step 2: Plot -----


In [ ]:
# %% [Task 1b — Population pyramid by age group and sex]
# Principle 2: one trace per series (Male / Female).
# Principle 4: barmode='relative' makes the two sides mirror into a pyramid,
# once we flip the male counts to negative.

# ----- Step 1: Prepare -----
# AGE_GROUP is already a clean 5-year band; list it youngest -> oldest.


# ----- Step 2: Plot -----


# Build symmetric ticks that show ABSOLUTE counts on both sides of zero.


In [ ]:
# %% [Task 2 — People by population group]
# Principle 2: one trace per visual series. Principle 5: prepare first, plot second.

# ----- Step 1: Prepare -----

# ----- Step 2: Plot -----


In [ ]:
# %% [Task 3 — Households by province]
# Principle 2 again. Several provinces, so we use a vertical bar with rotated labels.

# ----- Step 1: Prepare -----

# ----- Step 2: Plot -----


In [ ]:
# %% [Task 4 — Household size distribution (donut)]
# Principle 1: a pie is still one trace + layout. Use pies only with 2–4 categories,
# so we bucket the numeric household size into 4 bands first.

# ----- Step 1: Prepare -----


# ----- Step 2: Plot -----


In [ ]:
# %% [Task 5 — Household landscape, 2x2 subplots]
# Principle 3: several things in one figure = several traces, via make_subplots.

# ----- Step 1: Prepare -----


# ----- Step 2: Plot -----


In [ ]:
# %% [Task 6 — Household head age: distribution + box by sex]
# Principle 3: mixed chart types in one figure. Principle 5: prepare first, plot second.

# ----- Step 1: Prepare -----

# ----- Step 2: Plot -----


In [ ]:
# %% [Task 7 — Tenure composition by population group, stacked bar]
# Principle 4: barmode='stack' lives in the layout.
# crosstab + normalize turns counts into shares that sum to 100% per group.
# No merge needed — tenure and population group both live in the Households file.


In [ ]:
# %% [Task 8 — Basic-services heatmap by province] (advanced — uses a merge)
# Principle 1: one Heatmap trace + layout. Principle 5: all the work is in the prep.
# Both files are one row per household, so this is a safe one-to-one merge on QID.


In [ ]:
# %% [Task 9 — Diverging bar: frequency of adult hunger]
# Principle 2: one trace per Likert level.
# Principle 4: barmode='relative' is the layout setting that creates the diverging effect.

# Homework: replicate this for A5_CHILD_HUNGER (drop the
# 'Not applicable (no child in the household)' responses first).

In [ ]:
# %% [Task 10 — Internet access by population group, grouped bar]
# Principle 2: one trace per group.


In [ ]:
# %% [Task 11 — Ownership of household assets, stacked Yes/No]
# Principle 2 + Principle 5: same recipe applied to several asset columns.


In [ ]:
# %% [Task 12 — Household size vs head age, scatter by population group]
# Principle 1: one Scatter trace per group. Both axes are linear here — unlike sales,
# household size and age don't span orders of magnitude, so no log transform is needed.


In [ ]:
# %% [Capstone (optional) — services heatmap + hunger Likert in one figure]
# Nothing new conceptually: make_subplots + add_trace(..., row=, col=).
# Reuses heatmap_data from Task 8 and pct from Task 9.
